# 🔬 Biometric Facial Fatigue Screening System (Student Zombie Meter)
### โครงงานการประยุกต์ใช้การเรียนรู้ของเครื่อง (Machine Learning Application - 100 คะแนนเต็ม)
**รหัสนักศึกษา:** 6710210312
**หัวข้อ:** การตรวจคัดกรองความเหนื่อยล้าสะสมจากการพักผ่อนไม่เพียงพอด้วยเทคนิคคอมพิวเตอร์วิทัศน์และการเรียนรู้ของเครื่อง (Scikit-Learn)

---
#### 📋 สารบัญขั้นตอนการทำงานใน Notebook:
1. การนำเข้าไลบรารีที่จำเป็น (Libraries Import)
2. การสร้างและจำลองชุดข้อมูลชีวมิติ 7 ฟีเจอร์ (Synthetic Biometric Dataset)
3. การประมวลผลข้อมูลขั้นสูงครบ 6 ด้าน (Data Preprocessing):
   - 3.1 การจัดการข้อมูลสูญหาย (Missing Value Imputation ด้วย Median)
   - 3.2 การจัดการข้อมูลซ้ำซ้อน (Duplicate Rows Removal)
   - 3.3 การแปลงตัวแปรเชิงหมวดหมู่ (One-Hot Encoding)
   - 3.4 การแบ่งชุดข้อมูล (Stratified Train/Test Split 80:20)
   - 3.5 การคัดเลือกฟีเจอร์เด่น (Feature Selection ด้วย SelectKBest ANOVA F-test)
   - 3.6 การปรับสเกลข้อมูล (StandardScaler Fit เฉพาะบน Train Set ป้องกัน Data Leakage)
4. การสร้างและปรับจูนโมเดลด้วย GridSearchCV (Model Training & Hyperparameter Tuning)
   - Logistic Regression (Tuned)
   - Support Vector Machine (Tuned)
   - Random Forest Classifier (Tuned)
5. การประเมินผลและเปรียบเทียบโมเดล (Model Evaluation & Comparison)
6. การวิเคราะห์ความสำคัญของฟีเจอร์ (Feature Importance Analysis)
7. การทดสอบการทำนายข้อมูลรายบุคคล (Single Sample Diagnostic Inference)



In [ ]:
# 1. นำเข้าไลบรารีมาตรฐาน
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# ตั้งค่าการแสดงผลกราฟ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("✅ นำเข้าไลบรารีสำเร็จเรียบร้อย!")


## 2. การสร้างและจำลองชุดข้อมูลชีวมิติ (Dataset Generation)
จำลองข้อมูลขนาด 1,500 ตัวอย่าง ประกอบด้วย:
- **5 Numerical Features:** `eye_openness`, `eye_aspect_ratio` (EAR), `mouth_aspect_ratio` (MAR), `under_eye_darkness_ratio`, `skin_texture_var`
- **2 Categorical Features:** `lighting_condition` (Well-Lit, Fluorescent, Dim-Light), `time_slot` (Daytime, Evening, Overnight)
- **Data Cleaning Simulation:** มี Missing Values 10 จุด และ Duplicate Rows 5 แถว เพื่อสาธิตการคลีนข้อมูลตามเกณฑ์อาจารย์


In [ ]:
def generate_dataset(n_samples=1500, random_state=42):
    np.random.seed(random_state)
    n_per_class = n_samples // 2

    # กลุ่มสดชื่น (Fresh - Class 0)
    eye_openness_0 = np.random.normal(0.75, 0.12, n_per_class)
    ear_0 = np.random.normal(0.34, 0.05, n_per_class)
    mar_0 = np.random.normal(0.23, 0.05, n_per_class)
    darkness_0 = np.random.normal(0.96, 0.08, n_per_class)
    texture_0 = np.random.normal(0.70, 0.13, n_per_class)
    lighting_0 = np.random.choice(['Well-Lit', 'Fluorescent', 'Dim-Light'], size=n_per_class, p=[0.6, 0.3, 0.1])
    time_0 = np.random.choice(['Daytime', 'Evening', 'Overnight'], size=n_per_class, p=[0.7, 0.25, 0.05])

    # กลุ่มเหนื่อยล้า (Fatigued - Class 1)
    eye_openness_1 = np.random.normal(0.47, 0.13, n_per_class)
    ear_1 = np.random.normal(0.23, 0.05, n_per_class)
    mar_1 = np.random.normal(0.40, 0.14, n_per_class)
    darkness_1 = np.random.normal(0.82, 0.09, n_per_class)
    texture_1 = np.random.normal(0.48, 0.14, n_per_class)
    lighting_1 = np.random.choice(['Well-Lit', 'Fluorescent', 'Dim-Light'], size=n_per_class, p=[0.2, 0.3, 0.5])
    time_1 = np.random.choice(['Daytime', 'Evening', 'Overnight'], size=n_per_class, p=[0.1, 0.3, 0.6])

    eye_openness = np.concatenate([eye_openness_0, eye_openness_1])
    ear = np.concatenate([ear_0, ear_1])
    mar = np.concatenate([mar_0, mar_1])
    darkness = np.concatenate([darkness_0, darkness_1])
    texture = np.concatenate([texture_0, texture_1])
    lighting = np.concatenate([lighting_0, lighting_1])
    time_slot = np.concatenate([time_0, time_1])
    target = np.array([0] * n_per_class + [1] * n_per_class)

    # แทรก Realistic Noise 2% เพื่อสะท้อนความแปรปรวนในโลกจริง
    flip_indices = np.random.choice(n_samples, size=int(n_samples * 0.02), replace=False)
    target[flip_indices] = 1 - target[flip_indices]

    df = pd.DataFrame({
        'eye_openness': np.round(np.clip(eye_openness, 0.1, 1.0), 4),
        'eye_aspect_ratio': np.round(np.clip(ear, 0.08, 0.50), 4),
        'mouth_aspect_ratio': np.round(np.clip(mar, 0.10, 0.85), 4),
        'under_eye_darkness_ratio': np.round(np.clip(darkness, 0.50, 1.15), 4),
        'skin_texture_var': np.round(np.clip(texture, 0.10, 1.0), 4),
        'lighting_condition': lighting,
        'time_slot': time_slot,
        'target': target
    })

    # แทรก Missing Values 10 จุด
    missing_indices = np.random.choice(df.index, size=10, replace=False)
    df.loc[missing_indices, 'skin_texture_var'] = np.nan

    # แทรก Duplicate Rows 5 แถว
    duplicate_rows = df.iloc[:5].copy()
    df = pd.concat([df, duplicate_rows], ignore_index=True)
    return df.sample(frac=1, random_state=random_state).reset_index(drop=True)

df_raw = generate_dataset()
print(f"ขนาดชุดข้อมูลดิบ: {df_raw.shape[0]} แถว, {df_raw.shape[1]} คอลัมน์")
display(df_raw.head())


## 3. กระบวนการ Data Preprocessing (เกณฑ์ 15 คะแนนเต็ม ครบ 6 มิติ)
ครอบคลุม 6 มิติตามข้อกำหนดวิชา:
1. **Missing Value Imputation:** แทนที่ค่าสูญหายด้วย Median
2. **Duplicate Removal:** ลบแถวซ้ำซ้อน
3. **One-Hot Encoding:** แปลงตัวแปร Category ให้เป็นตัวเลข
4. **Stratified Train/Test Split (80:20):** ป้องกันคลาสเอียง
5. **Feature Selection:** ใช้ `SelectKBest` (ANOVA F-test) บน Train Set
6. **Feature Scaling:** ปรับสเกลด้วย `StandardScaler` (Fit เฉพาะ Train Set ป้องกัน Data Leakage)


In [ ]:
# 3.1 การจัดการ Missing Values
print("1. ตรวจสอบ Missing Values เริ่มต้น:")
print(df_raw.isnull().sum())
df_clean = df_raw.copy()
median_val = df_clean['skin_texture_var'].median()
df_clean['skin_texture_var'] = df_clean['skin_texture_var'].fillna(median_val)
print(f"-> ทำการแทนที่ Missing ด้วย Median ({median_val:.4f}) สำเร็จ! คงเหลือ: {df_clean.isnull().sum().sum()} จุด\n")

# 3.2 การจัดการ Duplicate Rows
duplicates_found = df_clean.duplicated().sum()
print(f"2. ตรวจพบข้อมูลซ้ำ: {duplicates_found} แถว")
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"-> ลบข้อมูลซ้ำเรียบร้อย ขนาดข้อมูลคงเหลือ: {len(df_clean)} แถว\n")

# 3.3 One-Hot Encoding ตัวแปรเชิงกลุ่ม
print("3. ทำ One-Hot Encoding ตัวแปรเชิงกลุ่ม (lighting_condition, time_slot):")
df_encoded = pd.get_dummies(df_clean, columns=['lighting_condition', 'time_slot'], drop_first=False, dtype=float)
print(f"-> จำนวนคอลัมน์หลัง Encoding: {df_encoded.shape[1]} คอลัมน์")

X = df_encoded.drop(columns=['target'])
y = df_encoded['target']
all_features = list(X.columns)

# 3.4 Stratified Train/Test Split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"4. แบ่งชุดข้อมูล: Train Set = {len(X_train)} ตัวอย่าง, Test Set = {len(X_test)} ตัวอย่าง\n")

# 3.5 Feature Selection ด้วย SelectKBest (ANOVA F-test)
# Fit บน Train Set เท่านั้น เพื่อป้องกัน Data Leakage
selector = SelectKBest(score_func=f_classif, k=8)
selector.fit(X_train, y_train)
selected_feature_cols = [col for col, sel in zip(all_features, selector.get_support()) if sel]
print(f"5. Feature Selection: คัดเลือก {len(selected_feature_cols)} ฟีเจอร์เด่น:")
for i, col in enumerate(selected_feature_cols, 1):
    print(f"   {i}. {col}")

X_train_selected = X_train[selected_feature_cols]
X_test_selected = X_test[selected_feature_cols]

# 3.6 Feature Scaling ด้วย StandardScaler (Fit เฉพาะ Train Set)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)
print("\n6. ปรับสเกลข้อมูลสำเร็จด้วย StandardScaler (Fit on Train only) ป้องกัน Data Leakage 100%!")


## 4. การพัฒนาโมเดลและการปรับจูน Hyperparameter ด้วย GridSearchCV (เกณฑ์ 20 คะแนน)
เปรียบเทียบโมเดล Scikit-Learn 3 ประเภท พร้อมปรับจูนพารามิเตอร์ด้วย **5-Fold Stratified Cross Validation**:
1. **Logistic Regression (Tuned):** ปรับค่า $C$
2. **Support Vector Machine (Tuned):** ปรับค่า $C$ และ $\gamma$
3. **Random Forest Classifier (Tuned):** ปรับค่า $n\_estimators$ และ $max\_depth$


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("⚙️ 1. กำลังปรับจูน Logistic Regression...")
lr_param_grid = {'C': [0.1, 1.0, 10.0]}
grid_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=500),
                       lr_param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid_lr.fit(X_train_scaled, y_train)

print("⚙️ 2. กำลังปรับจูน Support Vector Machine (RBF Kernel)...")
svm_param_grid = {'C': [0.1, 1.0, 10.0], 'gamma': ['scale', 'auto']}
grid_svm = GridSearchCV(SVC(kernel='rbf', probability=True, random_state=42),
                        svm_param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train_scaled, y_train)

print("⚙️ 3. กำลังปรับจูน Random Forest Classifier...")
rf_param_grid = {'n_estimators': [50, 100], 'max_depth': [3, 5, 8]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42),
                       rf_param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train_scaled, y_train)

models = {
    'Logistic Regression (Tuned)': grid_lr.best_estimator_,
    'Support Vector Machine (Tuned)': grid_svm.best_estimator_,
    'Random Forest (Tuned)': grid_rf.best_estimator_
}
print("\n✅ การปรับจูน Hyperparameter สำเร็จครบทั้ง 3 โมเดล!")


## 5. การประเมินผลและเปรียบเทียบโมเดล (เกณฑ์ 15 คะแนน)
วัดผลด้วยเมทริกซ์มาตรฐานครบถ้วน:
- Train Accuracy, Test Accuracy
- Precision, Recall, F1-Score, ROC-AUC
- 5-Fold Cross Validation F1 Score (Mean ± Std)
- Confusion Matrix Heatmaps สำหรับเปรียบเทียบทั้ง 3 โมเดล


In [ ]:
results = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, model) in enumerate(models.items()):
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)
    test_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, test_pred)
    prec = precision_score(y_test, test_pred)
    rec = recall_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred)
    roc_auc = roc_auc_score(y_test, test_proba)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1')

    results.append({
        'Model': name,
        'Train Acc': round(accuracy_score(y_train, train_pred), 4),
        'Test Acc': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        '5-Fold CV F1': f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}"
    })

    cm = confusion_matrix(y_test, test_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Fresh (0)', 'Fatigued (1)'],
                yticklabels=['Fresh (0)', 'Fatigued (1)'])
    axes[idx].set_title(f"{name}\nTest F1: {f1:.4f}")
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(results)
print("🏆 ตารางสรุปผลการเปรียบเทียบโมเดล:")
display(results_df)


## 6. การวิเคราะห์ความสำคัญของฟีเจอร์ (Feature Importance Analysis)
วิเคราะห์ลำดับความสำคัญของสัญญาณทางชีวมิติและตัวแปรบริบทผ่าน Random Forest เพื่อให้โมเดลสามารถอธิบายผลลัพธ์ได้อย่างโปร่งใส (Explainable AI)


In [ ]:
rf_model = models['Random Forest (Tuned)']
feat_df = pd.DataFrame({
    'Feature': selected_feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 4.5))
sns.barplot(x='Importance', y='Feature', data=feat_df, hue='Feature', palette='viridis', legend=False)
plt.title('Feature Importance (Selected Biometric & Contextual Features)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

display(feat_df)


## 7. การบันทึกโมเดลและทดสอบการวินิจฉัยข้อมูลเดี่ยว (Deployment Preparation)
บันทึก Artifacts สำคัญสำหรับนำไปใช้บน Streamlit Web Application:
- `best_model.joblib`: โมเดลที่ดีที่สุด
- `scaler.joblib`: Scaler ที่ผ่านการ Fit
- `feature_cols.joblib`: รายชื่อฟีเจอร์ที่ได้รับเลือก


In [ ]:
os.makedirs('models', exist_ok=True)
best_row = results_df.sort_values(by='F1-Score', ascending=False).iloc[0]
best_name = best_row['Model']
best_model = models[best_name]

joblib.dump(best_model, 'models/best_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')
joblib.dump(selected_feature_cols, 'models/feature_cols.joblib')
print(f"💾 บันทึกโมเดลสำเร็จ! โมเดลที่ได้รับเลือก: {best_name}")

# ทดสอบจำลองการวินิจฉัยคนที่มีอาการเหนื่อยล้า (Single Inference Test)
sample_input = {
    'eye_openness': 0.42,
    'eye_aspect_ratio': 0.20,
    'mouth_aspect_ratio': 0.45,
    'under_eye_darkness_ratio': 0.78,
    'skin_texture_var': 0.40,
    'lighting_condition': 'Dim-Light',
    'time_slot': 'Overnight'
}

sample_df = pd.DataFrame([sample_input])
sample_encoded = pd.get_dummies(sample_df, columns=['lighting_condition', 'time_slot'], dtype=float)
sample_full = sample_encoded.reindex(columns=selected_feature_cols, fill_value=0.0)
sample_scaled = scaler.transform(sample_full)

pred = best_model.predict(sample_scaled)[0]
prob = best_model.predict_proba(sample_scaled)[0][1]

status = '⚠️ Fatigued (เหนื่อยล้าสะสม/อดนอน)' if pred == 1 else '✅ Fresh (ร่างกายสดชื่นพร้อมทำงาน)'
print(f"ผลการทดสอบจำลอง: {status}")
print(f"ระดับความน่าจะเป็นของความเหนื่อยล้า (Fatigue Probability): {prob * 100:.2f}%")
print(f"ดัชนีความตื่นตัว (Alertness Index): {(1.0 - prob) * 100:.2f}%")
